<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Document Metadata

**Add metadata to documents as you upload them. Search a large library by its metadata.**

Metadata carries what a file does not say about itself. An SEC filing's file name, such as `nvda-20240728.pdf`, gives the company's ticker and the date its period ended, but not the form or the fiscal quarter, and fiscal calendars differ: NVIDIA's second quarter of fiscal 2025 ended July 28, 2024, while AMD's ended June 28, 2025. This notebook demonstrates how to create a collection on PageIndex Cloud, declare its metadata fields, add metadata to documents as you upload them, and have the chat agent filter and count documents by it.

The notebook has two parts:

- **Part 1 · Add metadata on upload** — With your own PageIndex API key, create a collection, declare its metadata fields, and upload two 10-Q filings into it with `company`, `form_type`, `fiscal_year`, and `fiscal_quarter` metadata.
- **Part 2 · Search a large library by metadata** — With a read-only demo key, ask a collection of 120 filings with the same metadata fields, and see the agent filter and count them by metadata.

**Before you start:** Have your [PageIndex API key](https://developer.pageindex.ai/api-keys) and OpenAI API key ready. Part 1 creates a collection in your PageIndex Cloud library and uploads two filings into it.

---

# Part 1 · Add metadata on upload

This part uses **your own PageIndex API key**. It creates a collection in your library and uploads two filings into it.

## 01 · Set up

Install the latest PageIndex SDK. Requests downloads the filings.

In [ ]:
!pip install -U pageindex requests

### Configure your client

Set your PageIndex API key. The client stores your documents and their metadata on PageIndex Cloud.

In [1]:
import os
from pageindex import PageIndexClient

os.environ["PAGEINDEX_API_KEY"] = "Your PageIndex API key here"

client = PageIndexClient(index="cloud")

## 02 · Declare the fields

Create a collection and declare its fields. Each field has a type, `string`, `number`, or `boolean`, and a description that tells the agent what it holds. A collection's fields are set once; to change them, call `delete_metadata_schema(collection_id=...)` and declare them again. A collection without declared fields learns them from the metadata you upload. A collection left by an earlier run is reused. `get_metadata_schema(collection_id=...)` shows the fields the collection has.

In [2]:
COLLECTION = "Metadata Demo Filings"
FIELDS = {
    "company": {"type": "string", "description": "Filing company: NVIDIA, AMD, Intel, Broadcom, Qualcomm, or Micron"},
    "form_type": {"type": "string", "description": "SEC form: 10-K (annual report) or 10-Q (quarterly report)"},
    "fiscal_year": {"type": "number", "description": "Fiscal year the company reports the filing under; the companies' fiscal years end in different months (NVIDIA's FY2025 ended Jan 26, 2025)"},
    "fiscal_quarter": {"type": "number", "description": "Fiscal quarter 1-3 of a 10-Q; a 10-K has none"},
}

existing = {c["name"]: c["id"] for c in client.list_collections()["collections"]}
if COLLECTION in existing:  # created on an earlier run
    collection_id = existing[COLLECTION]
else:
    collection_id = client.create_collection(COLLECTION)["collection"]["id"]
    client.create_metadata_schema(collection_id=collection_id, fields=FIELDS)
client.get_metadata_schema(collection_id=collection_id)["schema"]

{'fields': {'company': {'type': 'string',
   'description': 'Filing company: NVIDIA, AMD, Intel, Broadcom, Qualcomm, or Micron',
   'documents': 0,
   'values': []},
  'form_type': {'type': 'string',
   'description': 'SEC form: 10-K (annual report) or 10-Q (quarterly report)',
   'documents': 0,
   'values': []},
  'fiscal_year': {'type': 'number',
   'description': "Fiscal year the company reports the filing under; the companies' fiscal years end in different months (NVIDIA's FY2025 ended Jan 26, 2025)",
   'documents': 0,
   'min': None,
   'max': None},
  'fiscal_quarter': {'type': 'number',
   'description': 'Fiscal quarter 1-3 of a 10-Q; a 10-K has none',
   'documents': 0,
   'min': None,
   'max': None}}}

## 03 · Add metadata on upload

Download two 10-Q filings from the PageIndex repository and submit each one into the collection with its metadata. Both cover the second quarter of fiscal 2025, though their file names are almost a year apart; their metadata says so. `wait=True` returns once the document is ready. A filing already in the collection from an earlier run is reused rather than uploaded again.

In [3]:
import requests

BASE_URL = "https://raw.githubusercontent.com/VectifyAI/PageIndex/main/examples/documents/"
DOCS = {
    "nvda-20240728.pdf": {"company": "NVIDIA", "form_type": "10-Q", "fiscal_year": 2025, "fiscal_quarter": 2},
    "amd-20250628.pdf": {"company": "AMD", "form_type": "10-Q", "fiscal_year": 2025, "fiscal_quarter": 2},
}

existing = {doc["name"]: doc["id"] for doc in client.list_documents(collection_id=collection_id)["documents"]}

doc_ids = {}
for file_name, metadata in DOCS.items():
    if file_name in existing:  # uploaded by an earlier run
        doc_ids[file_name] = existing[file_name]
        print(f"{file_name}: {doc_ids[file_name]} (already uploaded)")
        continue

    response = requests.get(BASE_URL + file_name, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)

    doc_ids[file_name] = client.submit_document(
        file_name, wait=True, collection_id=collection_id, metadata=metadata,
    )["doc_id"]
    print(f"{file_name}: {doc_ids[file_name]}")

nvda-20240728.pdf: pi-cmwlvcayg000v09nvcji7iddk
amd-20250628.pdf: pi-cmwlvcjw4000w09nvsy2uwy5o


`get_document()` returns a document's `metadata`, and `list_documents()` returns it on every document.


In [4]:
for file_name, doc_id in doc_ids.items():
    print(f"{file_name}: {client.get_document(doc_id)['metadata']}")

nvda-20240728.pdf: {'company': 'NVIDIA', 'form_type': '10-Q', 'fiscal_year': 2025, 'fiscal_quarter': 2}
amd-20250628.pdf: {'company': 'AMD', 'form_type': '10-Q', 'fiscal_year': 2025, 'fiscal_quarter': 2}


Metadata is a flat dict of up to 16 `str`, `int`, `float`, or `bool` values (numbers must be finite, so no `NaN` or `Infinity`). A key is 1 to 64 characters and does not start with `$`; a string value is at most 512 characters. In a collection with declared fields, a document may use only those fields, each with its declared type. PageIndex Cloud rejects metadata that breaks a rule with `PageIndexAPIError`. See [Collections with metadata](https://pageindex.ai/docs/documents/organize) for details.


---

# Part 2 · Search a large library by metadata

A collection of 120 filings with the same four metadata fields is ready on PageIndex Cloud: the 10-K and 10-Q filings of NVIDIA, AMD, Intel, Broadcom, Qualcomm, and Micron, five fiscal years each, about 11,500 pages. Its files are named by EDGAR accession number. This part queries it with **a read-only demo key**, so nothing is uploaded.

## 04 · Connect to the library

Set your OpenAI API key and choose your chat `MODEL`. The `demo` client reads the library with the read-only demo key, and your chosen model answers with PageIndex Cloud's retrieval tools.

In [5]:
os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"

MODEL = "gpt-5.6-luna"  # gpt-5.6-sol / gpt-5.6-terra / gpt-5.6-luna

demo = PageIndexClient(
    index={"api_key": "pageindex_demo_readonly_key"},
    chat=MODEL,
)
LIBRARY_ID = "cmwluacj600090anvd8g8o04z"  # semiconductor-sec-filings

## 05 · Ask by metadata

Ask for the third-quarter 10-Qs from fiscal 2024 onward. The agent lists the collection with `list_documents`, 50 filings at a time. Each listed filing carries its metadata, so the agent picks out the matches by `form_type`, `fiscal_quarter`, and `fiscal_year` without opening a filing. The output streams the answer alongside retrieval tool calls.

In [6]:
messages = [{"role": "user", "content": "List the third-quarter 10-Qs from fiscal 2024 onward, with each company and fiscal year."}]

response = None
for event in demo.chat(messages, collection_id=LIBRARY_ID, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] list_documents({"collection":"semiconductor-sec-filings","limit":50})

[Tool call] list_documents({"collection":"semiconductor-sec-filings","offset":50,"limit":50})

[Tool call] list_documents({"collection":"semiconductor-sec-filings","offset":100,"limit":50})
Third-quarter 10-Qs from fiscal 2024 onward:

| Company | Fiscal year | Filing document |
|---|---:|---|
| AMD | 2024 | `0000002488-24-000163.pdf` |
| AMD | 2025 | `0000002488-25-000166.pdf` |
| Intel | 2024 | `0000050863-24-000149.pdf` |
| Intel | 2025 | `0000050863-25-000179.pdf` |
| Broadcom | 2024 | `0001730168-24-000099.pdf` |
| Broadcom | 2025 | `0001730168-25-000098.pdf` |
| Qualcomm | 2024 | `0000804328-24-000056.pdf` |
| Qualcomm | 2025 | `0000804328-25-000045.pdf` |
| Micron | 2024 | `0000723125-24-000019.pdf` |
| Micron | 2025 | `0000723125-25-000021.pdf` |
| Micron | 2026 | `0000723125-26-000015.pdf` |
| NVIDIA | 2024 | `0001045810-23-000227.pdf` |
| NVIDIA | 2025 | `0001045810-24-000316.pdf` |
| NVIDIA |

The agent can also count documents by metadata. Ask how many 10-Ks and 10-Qs each company has: the agent lists the collection's filings with their metadata and counts them by `company` and `form_type`, without opening a filing.

In [7]:
messages = [{"role": "user", "content": "How many 10-K and 10-Q filings does each company have?"}]

response = None
for event in demo.chat(messages, collection_id=LIBRARY_ID, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] list_documents({"collection":"semiconductor-sec-filings","limit":50})

[Tool call] list_documents({"collection":"semiconductor-sec-filings","offset":50,"limit":50})

[Tool call] list_documents({"collection":"semiconductor-sec-filings","offset":100,"limit":50})
| Company | 10-K filings | 10-Q filings | Total |
|---|---:|---:|---:|
| AMD | 5 | 15 | 20 |
| Intel | 5 | 15 | 20 |
| Broadcom | 5 | 15 | 20 |
| Qualcomm | 5 | 15 | 20 |
| Micron | 5 | 15 | 20 |
| NVIDIA | 5 | 15 | 20 |

The collection contains **120 filings total**: **30 10-Ks** and **90 10-Qs**.

## 06 · Review usage & estimated cost (optional)

Inspect the token usage for the last answer and estimate its model cost using the rate table below. Declaring fields, adding metadata, and reading it back do not require model calls. This estimate does not include PageIndex Cloud charges.


In [8]:
import json

# Standard USD per 1M tokens: (input, cached input, output).
# Checked 2026-09-17: https://developers.openai.com/api/docs/models/{model}
PRICES = {
    "gpt-5.6-sol": (4.0, 0.4, 20.0),
    "gpt-5.6-terra": (2.0, 0.2, 12.0),
    "gpt-5.6-luna": (0.2, 0.02, 1.2),
}
input_rate, cached_rate, output_rate = PRICES[MODEL]

usage = response.get("usage") if response else None
print("\n\nUsage:", json.dumps(usage, ensure_ascii=False, indent=2))
if response and response.get("status") != "completed":
    print("Response did not complete:", response.get("error") or response.get("incomplete_details"))

if usage:
    details = usage.get("input_tokens_details") or {}
    cached = details.get("cached_tokens", 0) or 0
    written = details.get("cache_write_tokens", 0) or 0
    regular = usage["input_tokens"] - cached - written
    cost = (regular * input_rate + cached * cached_rate + written * input_rate * 1.25
            + usage["output_tokens"] * output_rate) / 1_000_000
    print(f"Estimated model cost at base rates: ${cost:.6f} USD")
    if usage["input_tokens"] > 272_000:
        print("Usage is aggregated across model calls; the estimate excludes possible long-context surcharges.")
else:
    print("No usage returned; unable to estimate cost.")



Usage: {
  "input_tokens": 34104,
  "input_tokens_details": {
    "cached_tokens": 15380,
    "cache_write_tokens": 18715
  },
  "output_tokens": 360,
  "output_tokens_details": {
    "reasoning_tokens": 126
  },
  "total_tokens": 34464,
  "request_usage_entries": [
    {
      "input_tokens": 4842,
      "input_tokens_details": {
        "cached_tokens": 0,
        "cache_write_tokens": 4839
      },
      "output_tokens": 55,
      "output_tokens_details": {
        "reasoning_tokens": 27
      },
      "total_tokens": 4897
    },
    {
      "input_tokens": 10544,
      "input_tokens_details": {
        "cached_tokens": 4839,
        "cache_write_tokens": 5702
      },
      "output_tokens": 126,
      "output_tokens_details": {
        "reasoning_tokens": 52
      },
      "total_tokens": 10670
    },
    {
      "input_tokens": 18718,
      "input_tokens_details": {
        "cached_tokens": 10541,
        "cache_write_tokens": 8174
      },
      "output_tokens": 179,
      "out

---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
